# NB12: Model Debug — Qwen2.5-7B Anomaly

**Question:** Why does Qwen2.5-7B (7B params) perform worse than Qwen2.5-1.5B (1.5B params)?

This notebook investigates the Qwen2.5-7B anomaly identified in the `smart_retrieval_slm` study:
- Per-dataset x per-prompt F1 heatmaps (target vs reference models)
- Answer length distribution comparison
- Per-question failure analysis (top questions where target fails but refs succeed)
- Error type classification (refusal, over-verbose, wrong, partial)
- Prompt ranking by competitiveness gap

**Parameterized:** Change `TARGET_MODEL` and `REFERENCE_MODELS` to investigate any model.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

from analysis_utils import (
    load_all_results, setup_plotting, load_per_item_scores,
    PRIMARY_METRIC, BROKEN_MODELS, MODEL_PARAMS, MODEL_TIER,
    DATASET_COLORS, DATASET_LABELS,
)

setup_plotting()
STUDY_PATH = Path("../outputs/smart_retrieval_slm")

# === PARAMETERIZED ===
TARGET_MODEL = 'Qwen2.5-7B'
REFERENCE_MODELS = ['Mistral-7B', 'Gemma2-9B', 'Qwen2.5-3B', 'Qwen2.5-1.5B']
# ====================

df_all = load_all_results(STUDY_PATH)
df = df_all[~df_all['model_short'].isin(BROKEN_MODELS)].copy()

target_df = df[df['model_short'] == TARGET_MODEL]
ref_df = df[df['model_short'].isin(REFERENCE_MODELS)]

print(f"Target: {TARGET_MODEL} — {len(target_df)} experiments")
print(f"References: {REFERENCE_MODELS}")
for m in REFERENCE_MODELS:
    n = len(df[df['model_short'] == m])
    print(f"  {m}: {n} experiments")
print(f"\nTarget mean F1: {target_df[PRIMARY_METRIC].mean():.4f}")
print(f"Reference mean F1: {ref_df[PRIMARY_METRIC].mean():.4f}")

## 1. Per-Dataset x Per-Prompt F1 Heatmaps

Compare the target model's F1 across all (dataset, prompt) combinations against
reference models. If the target fails specifically on certain prompts, it suggests
a prompt format sensitivity rather than a fundamental capability issue.

In [ ]:
# Heatmaps: dataset x prompt, one per model
models_to_show = [TARGET_MODEL] + REFERENCE_MODELS
rag_df = df[df['exp_type'] == 'rag'].copy()

# Compute mean F1 per (model, dataset, prompt)
pivot_data = rag_df.groupby(['model_short', 'dataset', 'prompt'])[PRIMARY_METRIC].mean().reset_index()

n_models = len(models_to_show)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), sharey=True)
if n_models == 1:
    axes = [axes]

vmin = pivot_data[PRIMARY_METRIC].quantile(0.05)
vmax = pivot_data[PRIMARY_METRIC].quantile(0.95)

for ax, model in zip(axes, models_to_show):
    m_data = pivot_data[pivot_data['model_short'] == model]
    if m_data.empty:
        ax.set_title(f'{model}\n(no data)')
        continue
    pivot = m_data.pivot(index='prompt', columns='dataset', values=PRIMARY_METRIC)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=vmin, vmax=vmax, ax=ax, linewidths=0.5)
    params = MODEL_PARAMS.get(model, '?')
    mean_f1 = m_data[PRIMARY_METRIC].mean()
    title = f'{model} ({params}B)\nmean F1={mean_f1:.3f}'
    if model == TARGET_MODEL:
        title = f'>>> {title} <<<'
    ax.set_title(title)

plt.suptitle(f'F1 by Dataset x Prompt — {TARGET_MODEL} vs References', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 2. Answer Length Distribution

Over-verbose answers are a common failure mode — the model generates correct information
buried in a long response, but F1 penalizes the extra tokens. If the target model produces
significantly longer answers, this could explain the low F1 without the model being
fundamentally worse.

In [ ]:
# Load per-item scores with text for answer length analysis
scores = load_per_item_scores(STUDY_PATH, metric=PRIMARY_METRIC, include_text=True)
scores = scores[scores['model_short'].isin([TARGET_MODEL] + REFERENCE_MODELS)].copy()

if 'prediction' in scores.columns:
    scores['pred_len'] = scores['prediction'].str.len()
    scores['pred_words'] = scores['prediction'].str.split().str.len()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Violin plot: answer length by model
    models_order = [TARGET_MODEL] + REFERENCE_MODELS
    plot_df = scores[scores['model_short'].isin(models_order)].copy()

    # Character length
    sns.violinplot(data=plot_df, x='model_short', y='pred_len',
                   order=models_order, ax=axes[0], cut=0, inner='box')
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('Prediction Length (chars)')
    axes[0].set_title('Answer Length Distribution')
    axes[0].tick_params(axis='x', rotation=30)

    # Scatter: prediction length vs F1
    for model in models_order:
        sub = plot_df[plot_df['model_short'] == model]
        color = '#e74c3c' if model == TARGET_MODEL else None
        alpha = 0.6 if model == TARGET_MODEL else 0.2
        s = 15 if model == TARGET_MODEL else 5
        axes[1].scatter(sub['pred_len'], sub[PRIMARY_METRIC],
                        s=s, alpha=alpha, label=model, color=color)

    axes[1].set_xlabel('Prediction Length (chars)')
    axes[1].set_ylabel('F1')
    axes[1].set_title('Length vs F1 — Is verbosity the issue?')
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary statistics
    print("\nAnswer Length Statistics (median characters):")
    len_stats = plot_df.groupby('model_short')['pred_len'].agg(['median', 'mean', 'std'])
    display(len_stats.loc[models_order].round(1))
else:
    print("No prediction text available. Ensure predictions.json has 'prediction' field.")

## 3. Per-Question Failure Analysis

Identify the top 20 questions where the target model fails (F1 < 0.1) but reference
models succeed (mean F1 > 0.3). Show actual predictions to diagnose failure patterns.

In [ ]:
if not scores.empty and 'prediction' in scores.columns:
    # Per-question mean F1 for target vs references
    target_scores = scores[scores['model_short'] == TARGET_MODEL].copy()
    ref_scores = scores[scores['model_short'].isin(REFERENCE_MODELS)].copy()

    target_q = target_scores.groupby(['idx', 'dataset']).agg(
        target_f1=(PRIMARY_METRIC, 'mean'),
        target_n=(PRIMARY_METRIC, 'count'),
    ).reset_index()

    ref_q = ref_scores.groupby(['idx', 'dataset']).agg(
        ref_f1=(PRIMARY_METRIC, 'mean'),
        ref_n=(PRIMARY_METRIC, 'count'),
    ).reset_index()

    q_compare = target_q.merge(ref_q, on=['idx', 'dataset'], how='inner')
    q_compare['gap'] = q_compare['ref_f1'] - q_compare['target_f1']

    # Top 20 where target fails but refs succeed
    failures = q_compare[
        (q_compare['target_f1'] < 0.1) & (q_compare['ref_f1'] > 0.3)
    ].nlargest(20, 'gap')

    print(f"Questions where {TARGET_MODEL} fails but references succeed:")
    print(f"  Total such questions: {len(q_compare[(q_compare['target_f1'] < 0.1) & (q_compare['ref_f1'] > 0.3)])}")
    print(f"  Showing top 20 by gap:\n")

    for _, row in failures.iterrows():
        idx = row['idx']
        ds = row['dataset']
        # Get sample predictions
        target_preds = target_scores[
            (target_scores['idx'] == idx) & (target_scores['dataset'] == ds)
        ]
        ref_preds = ref_scores[
            (ref_scores['idx'] == idx) & (ref_scores['dataset'] == ds)
        ]

        question = target_preds['question'].iloc[0] if 'question' in target_preds.columns and len(target_preds) > 0 else '?'
        expected = target_preds['expected'].iloc[0] if 'expected' in target_preds.columns and len(target_preds) > 0 else '?'

        print(f"  Q{idx} [{ds}] (gap={row['gap']:.3f})")
        print(f"    Question: {str(question)[:120]}")
        print(f"    Expected: {str(expected)[:80]}")
        # Show a target prediction sample
        if len(target_preds) > 0:
            sample_pred = target_preds.iloc[0]
            print(f"    {TARGET_MODEL} prediction: {str(sample_pred.get('prediction', ''))[:120]}")
            print(f"    {TARGET_MODEL} F1: {sample_pred[PRIMARY_METRIC]:.3f}")
        # Show a reference prediction sample
        if len(ref_preds) > 0:
            best_ref = ref_preds.nlargest(1, PRIMARY_METRIC).iloc[0]
            print(f"    Best ref ({best_ref['model_short']}) prediction: {str(best_ref.get('prediction', ''))[:120]}")
            print(f"    Best ref F1: {best_ref[PRIMARY_METRIC]:.3f}")
        print()
else:
    print("Per-item scores not available.")

## 4. Error Type Classification

Classify the target model's errors into categories:
- **Refusal**: Model refuses to answer ("I cannot", "I don't know", empty)
- **Over-verbose**: Answer is >200 chars (likely buried correct info)
- **Wrong answer**: Short but incorrect
- **Partial match**: F1 between 0.1 and 0.5 (got some tokens right)

In [ ]:
if not scores.empty and 'prediction' in scores.columns:
    target_all = scores[scores['model_short'] == TARGET_MODEL].copy()

    def classify_error(row):
        pred = str(row.get('prediction', ''))
        f1 = row.get(PRIMARY_METRIC, 0)
        if f1 >= 0.5:
            return 'correct'
        pred_lower = pred.lower().strip()
        # Refusal detection
        refusal_phrases = ['i cannot', 'i can\'t', 'i don\'t know', 'i am unable',
                           'sorry', 'not enough information', 'no answer',
                           'cannot determine', 'unable to']
        if not pred_lower or len(pred_lower) < 3:
            return 'refusal (empty)'
        if any(phrase in pred_lower for phrase in refusal_phrases):
            return 'refusal (explicit)'
        if 0.1 <= f1 < 0.5:
            return 'partial match'
        if len(pred) > 200:
            return 'over-verbose'
        return 'wrong answer'

    target_all['error_type'] = target_all.apply(classify_error, axis=1)

    # Distribution
    error_dist = target_all['error_type'].value_counts()
    error_pct = target_all['error_type'].value_counts(normalize=True)

    print(f"Error Classification for {TARGET_MODEL}:")
    print("=" * 50)
    for err_type in error_dist.index:
        print(f"  {err_type:<25s}: {error_dist[err_type]:>5d} ({error_pct[err_type]:.1%})")

    # Compare error distribution against references
    ref_all = scores[scores['model_short'].isin(REFERENCE_MODELS)].copy()
    ref_all['error_type'] = ref_all.apply(classify_error, axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Target pie chart
    target_counts = target_all['error_type'].value_counts()
    colors_map = {
        'correct': '#2ecc71', 'partial match': '#f39c12',
        'wrong answer': '#e74c3c', 'over-verbose': '#9b59b6',
        'refusal (empty)': '#95a5a6', 'refusal (explicit)': '#7f8c8d',
    }
    pie_colors = [colors_map.get(t, '#bdc3c7') for t in target_counts.index]
    axes[0].pie(target_counts.values, labels=target_counts.index,
                colors=pie_colors, autopct='%1.0f%%', startangle=90)
    axes[0].set_title(f'{TARGET_MODEL} Error Types')

    # Comparison bar chart (per model)
    all_models_err = pd.concat([target_all, ref_all])
    err_by_model = all_models_err.groupby('model_short')['error_type'].value_counts(normalize=True).unstack(fill_value=0)
    models_order = [TARGET_MODEL] + REFERENCE_MODELS
    err_by_model = err_by_model.reindex(models_order).fillna(0)
    err_by_model.plot(kind='bar', stacked=True, ax=axes[1],
                       color=[colors_map.get(c, '#bdc3c7') for c in err_by_model.columns])
    axes[1].set_ylabel('Fraction')
    axes[1].set_title('Error Type Distribution by Model')
    axes[1].legend(fontsize=7, loc='upper right')
    axes[1].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()

    # Per-dataset error breakdown for target
    print(f"\n{TARGET_MODEL} Error Types by Dataset:")
    for ds in sorted(target_all['dataset'].unique()):
        ds_sub = target_all[target_all['dataset'] == ds]
        ds_dist = ds_sub['error_type'].value_counts(normalize=True)
        top_errors = ', '.join(f"{k}={v:.0%}" for k, v in ds_dist.head(3).items())
        print(f"  {ds}: {top_errors}")
else:
    print("Prediction text not available for error classification.")

## 5. Per-Dataset F1 Gap Analysis

Quantify the competitiveness gap between the target and references across datasets,
to determine whether the anomaly is universal or dataset-specific.

In [ ]:
# Per-dataset comparison: target vs each reference
datasets = sorted(df['dataset'].unique())
models_all = [TARGET_MODEL] + REFERENCE_MODELS

# Mean F1 per (model, dataset, exp_type)
summary = df.groupby(['model_short', 'dataset', 'exp_type'])[PRIMARY_METRIC].agg(
    ['mean', 'std', 'count', 'max']
).reset_index()

fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 5), sharey=True)
if len(datasets) == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    ds_data = summary[(summary['dataset'] == ds) & (summary['exp_type'] == 'rag')]
    ds_data = ds_data[ds_data['model_short'].isin(models_all)]
    if ds_data.empty:
        ax.set_title(DATASET_LABELS.get(ds, ds))
        continue

    ds_data = ds_data.set_index('model_short').reindex(models_all)
    colors = ['#e74c3c' if m == TARGET_MODEL else '#3498db' for m in models_all]
    bars = ax.bar(range(len(models_all)), ds_data['mean'].values,
                  yerr=ds_data['std'].values, capsize=3,
                  color=colors, edgecolor='black', linewidth=0.5, alpha=0.8)
    # Add max F1 as diamond markers
    ax.scatter(range(len(models_all)), ds_data['max'].values,
               marker='D', color='gold', edgecolors='black', s=50, zorder=5, label='Best config')

    ax.set_xticks(range(len(models_all)))
    ax.set_xticklabels(models_all, rotation=30, ha='right', fontsize=9)
    ax.set_title(DATASET_LABELS.get(ds, ds))
    ax.set_ylabel('F1' if ax is axes[0] else '')
    ax.grid(axis='y', alpha=0.3)
    if ax is axes[0]:
        ax.legend(fontsize=8)

plt.suptitle(f'{TARGET_MODEL} (red) vs References (blue) — RAG Experiments', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# Numeric gap table
print(f"\nF1 Gap: {TARGET_MODEL} vs References (RAG experiments):")
print("=" * 70)
for ds in datasets:
    target_mean = summary[
        (summary['model_short'] == TARGET_MODEL) & (summary['dataset'] == ds) & (summary['exp_type'] == 'rag')
    ]['mean'].values
    if len(target_mean) == 0:
        continue
    target_mean = target_mean[0]
    print(f"\n  {ds}:")
    print(f"    {TARGET_MODEL}: {target_mean:.4f}")
    for ref in REFERENCE_MODELS:
        ref_mean = summary[
            (summary['model_short'] == ref) & (summary['dataset'] == ds) & (summary['exp_type'] == 'rag')
        ]['mean'].values
        if len(ref_mean) > 0:
            gap = ref_mean[0] - target_mean
            print(f"    {ref}: {ref_mean[0]:.4f} (gap={gap:+.4f})")

## 6. Is There a Prompt That Fixes It?

For each prompt, compute the **competitiveness gap**: how close is the target model
to the reference models? A prompt that "fixes" the target should have a small gap.
If no prompt fixes it, the issue is likely model-level (tokenizer, chat template, etc.).

In [ ]:
rag_all = df[df['exp_type'] == 'rag'].copy()

# Per (model, prompt, dataset) mean F1
prompt_perf = rag_all.groupby(['model_short', 'prompt', 'dataset'])[PRIMARY_METRIC].mean().reset_index()

# For each prompt × dataset, compute target F1 and mean reference F1
prompts = sorted(rag_all['prompt'].unique())
datasets = sorted(rag_all['dataset'].unique())

gap_rows = []
for prompt in prompts:
    for ds in datasets:
        target_f1 = prompt_perf[
            (prompt_perf['model_short'] == TARGET_MODEL) &
            (prompt_perf['prompt'] == prompt) &
            (prompt_perf['dataset'] == ds)
        ][PRIMARY_METRIC].values
        ref_f1 = prompt_perf[
            (prompt_perf['model_short'].isin(REFERENCE_MODELS)) &
            (prompt_perf['prompt'] == prompt) &
            (prompt_perf['dataset'] == ds)
        ][PRIMARY_METRIC].values
        if len(target_f1) > 0 and len(ref_f1) > 0:
            gap_rows.append({
                'prompt': prompt, 'dataset': ds,
                'target_f1': target_f1[0], 'ref_mean_f1': ref_f1.mean(),
                'gap': ref_f1.mean() - target_f1[0],
            })

gap_df = pd.DataFrame(gap_rows)

if not gap_df.empty:
    # Dataset-stratified gap per prompt
    prompt_gap = gap_df.groupby('prompt').agg(
        mean_gap=('gap', 'mean'),
        target_f1=('target_f1', 'mean'),
        ref_f1=('ref_mean_f1', 'mean'),
    ).sort_values('mean_gap')

    print(f"Prompt Ranking by Competitiveness Gap ({TARGET_MODEL} vs references):")
    print(f"{'Prompt':<25s} {'Target F1':>10s} {'Ref F1':>10s} {'Gap':>10s}")
    print("=" * 60)
    for prompt, row in prompt_gap.iterrows():
        indicator = ' <<<' if row['mean_gap'] == prompt_gap['mean_gap'].min() else ''
        print(f"  {prompt:<25s} {row['target_f1']:>8.4f} {row['ref_f1']:>8.4f} "
              f"{row['mean_gap']:>+8.4f}{indicator}")

    # Heatmap: gap by prompt x dataset
    gap_pivot = gap_df.pivot(index='prompt', columns='dataset', values='gap')
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(gap_pivot, annot=True, fmt='.3f', cmap='RdYlGn_r',
                center=0, ax=ax, linewidths=0.5)
    ax.set_title(f'Competitiveness Gap by Prompt x Dataset\n'
                 f'(positive = {TARGET_MODEL} underperforms references)')
    plt.tight_layout()
    plt.show()

    # Best prompt for the target
    best_prompt = prompt_gap['mean_gap'].idxmin()
    best_gap = prompt_gap.loc[best_prompt, 'mean_gap']
    print(f"\nBest prompt for {TARGET_MODEL}: '{best_prompt}' (gap={best_gap:+.4f})")
    if best_gap > 0.05:
        print(f"  Even the best prompt leaves a significant gap — issue is likely model-level.")
    else:
        print(f"  This prompt nearly closes the gap — the issue may be prompt sensitivity.")
else:
    print("Insufficient data for prompt gap analysis.")

## 7. Summary

**Diagnosis checklist for the Qwen2.5-7B anomaly:**
1. Is the issue prompt-specific or universal across all prompts?
2. Is it an answer length/verbosity issue (model generates correct info in verbose form)?
3. Is it a refusal issue (model refuses to answer with certain prompts)?
4. Is there any prompt that fixes the issue?
5. Is the issue dataset-specific or consistent across NQ/TriviaQA/HotpotQA?

**Next steps based on diagnosis:**
- If **prompt-fixable**: Add the working prompt to the focused study config
- If **verbosity issue**: Add post-processing to extract short answers
- If **model-level**: Add `Qwen2.5-7B` to `BROKEN_MODELS` and exclude from analysis